In [1]:
import pandas as pd
import heapq

In [2]:
df = pd.read_excel('dados/base_analise.xlsx')

## 1. Rede assistencial
* número de PESA (municípios com acesso local);
* municípios referenciados;
* tempo, distância medianos e percentis;
* faixas críticas.
* cobertura de pesa por região
* relação de numero de habitantes por pesa na região

In [3]:
# Número de PESA
df[df['ACESSO_LOCAL']==1].shape

(220, 61)

In [4]:
df.columns

Index(['Unnamed: 0', 'ACESSO_LOCAL', 'MULTIPLO_PESA', 'GVE', 'PESA', 'MUNI',
       'OBSERVACOES', 'LAT_MUNI', 'LON_MUNI', 'LAT_PESA', 'LON_PESA',
       'DISTANCIA', 'TEMPO', 'IBGE', 'POP10', 'POP11A59', 'POP60', 'POP_GERAL',
       'TOTAL_CASOS', 'TOTAL_10A', 'TOTAL_11A59', 'TOTAL_60', 'LEVE', 'MG',
       'MG_ORIGINAL', 'LEVE_10', 'MG_10', 'LEVE_11A59', 'MG_11A59', 'LEVE_60',
       'MG_60', 'TOTAL_AMPOLAS', 'OBITOS', 'OBITOS_10', 'OBITOS_11A59',
       'OBITOS_60', 'N_PESA', 'REGIAO', 'MUNI.1', '_merge', 'CAT_TEMPO',
       'PROP_POP_0A10', 'PROP_POP_60MAIS', 'LOG_POP', 'PROP_MG',
       'INCID_MEDIA_GERAL', 'INC_MEDIA_ANUAL', 'INCID_0A10', 'INCID_60MAIS',
       'PROP_CASOS_0A10', 'PROP_CASOS_60MAIS', 'TAXA_MG_100MIL', 'CLASS_IG',
       'SORO_IG', 'EVOL_IG', 'PROP_CLASS_IGN', 'PROP_SORO_IGN',
       'PROP_EVOL_IGN', 'BAIXO_N', 'TEMPO_50', 'CARGA_MG'],
      dtype='str')

In [5]:
# Número de municípios sem PESA
df[df['ACESSO_LOCAL']==0].shape[0]

425

In [6]:
# Tempo de acesso médio para municípios sem PESA
df[df['ACESSO_LOCAL']==0]['TEMPO'].describe().round(1)

count    425.0
mean      27.4
std       12.0
min        6.8
25%       19.0
50%       25.2
75%       34.3
max       70.8
Name: TEMPO, dtype: float64

In [7]:
# Tempo médio de acesso para municípios sem PESA por Região
df[df['ACESSO_LOCAL']==0].groupby('REGIAO')['TEMPO'].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
REGIAO,,,,,,,,
ARACATUBA,25.0,23.3,8.8,10.6,15.9,22.3,28.1,47.5
ARARAQUARA,17.0,30.6,11.3,15.6,22.7,28.5,37.2,54.1
BAIXADA SANTISTA,1.0,12.1,NaN,12.1,12.1,12.1,12.1,12.1
BARRETOS,12.0,30.6,11.4,15.7,23.8,29.9,35.8,57.6
BAURU,43.0,31.2,14.1,9.8,19.5,28.8,42.1,63.5
CAMPINAS,29.0,26.5,13.1,8.7,18.1,24.1,33.3,70.8
FRANCA,13.0,19.6,7.8,10.4,13.1,17.9,24.3,34.3
GRANDE SAO PAULO,23.0,22.2,9.3,7.4,15.5,21.0,27.6,45.9
MARILIA,54.0,32.3,12.5,8.5,22.9,28.8,41.6,61.6


In [8]:
# Municípios com Tempo crítico por Região
df.loc[(df['ACESSO_LOCAL']==0) & (df['TEMPO']>59),['REGIAO','MUNI','TEMPO']]

,REGIAO,MUNI,TEMPO
117,BAURU,PORANGABA,63.50
141,CAMPINAS,PIRACAIA,70.80
162,SOROCABA,BOM SUCESSO DE ITARARE,60.20
165,SOROCABA,RIBEIRA,59.10
328,SAO JOSE DO RIO,IRAPUA,60.70
392,TAUBATE,NATIVIDADE DA SERRA,63.70
429,MARILIA,UBIRAJARA,61.65


In [9]:
# Número de PESA, Municipio e População por Região

# - Número de PESA por Região
pesaReg = df[df['ACESSO_LOCAL']==1].groupby('REGIAO').size().reset_index(name='N_PESA')

# - Número de Municípios por Região
muniReg = df.groupby('REGIAO').size().reset_index(name='N_MUNICIPIOS')

# - População por Região
popReg = df.groupby('REGIAO')['POP_GERAL'].sum().reset_index(name='POP_REG')

# Merge Regiao, Pesa e contagens de Pesa
dfReg = pesaReg.merge(muniReg,on = 'REGIAO')

# Merge Regiao e População da Região
dfReg = dfReg.merge(popReg,on = 'REGIAO')


In [10]:
# Cobertura de Pesa por Região
dfReg['COB_PESA'] = (dfReg['N_PESA'] / dfReg['N_MUNICIPIOS']*100).round(1)


In [11]:
# População por Pesa
dfReg['POP_POR_PESA'] = (dfReg['POP_REG'] / dfReg['N_PESA']).round(1)


In [12]:
dfReg

,REGIAO,N_PESA,N_MUNICIPIOS,POP_REG,COB_PESA,POP_POR_PESA
0,ARACATUBA,15,40,778764,37.5,51917.6
1,ARARAQUARA,7,24,1017893,29.2,145413.3
2,BAIXADA SANTISTA,8,9,1850396,88.9,231299.5
3,BARRETOS,6,18,438357,33.3,73059.5
4,BAURU,25,68,1773654,36.8,70946.2
5,CAMPINAS,13,42,4733979,31.0,364152.2
6,FRANCA,9,22,700936,40.9,77881.8
7,GRANDE SAO PAULO,16,39,21498539,41.0,1343658.7
8,MARILIA,8,62,1133065,12.9,141633.1
9,PIRACICABA,9,26,1590372,34.6,176708.0


In [13]:
df.columns

Index(['Unnamed: 0', 'ACESSO_LOCAL', 'MULTIPLO_PESA', 'GVE', 'PESA', 'MUNI',
       'OBSERVACOES', 'LAT_MUNI', 'LON_MUNI', 'LAT_PESA', 'LON_PESA',
       'DISTANCIA', 'TEMPO', 'IBGE', 'POP10', 'POP11A59', 'POP60', 'POP_GERAL',
       'TOTAL_CASOS', 'TOTAL_10A', 'TOTAL_11A59', 'TOTAL_60', 'LEVE', 'MG',
       'MG_ORIGINAL', 'LEVE_10', 'MG_10', 'LEVE_11A59', 'MG_11A59', 'LEVE_60',
       'MG_60', 'TOTAL_AMPOLAS', 'OBITOS', 'OBITOS_10', 'OBITOS_11A59',
       'OBITOS_60', 'N_PESA', 'REGIAO', 'MUNI.1', '_merge', 'CAT_TEMPO',
       'PROP_POP_0A10', 'PROP_POP_60MAIS', 'LOG_POP', 'PROP_MG',
       'INCID_MEDIA_GERAL', 'INC_MEDIA_ANUAL', 'INCID_0A10', 'INCID_60MAIS',
       'PROP_CASOS_0A10', 'PROP_CASOS_60MAIS', 'TAXA_MG_100MIL', 'CLASS_IG',
       'SORO_IG', 'EVOL_IG', 'PROP_CLASS_IGN', 'PROP_SORO_IGN',
       'PROP_EVOL_IGN', 'BAIXO_N', 'TEMPO_50', 'CARGA_MG'],
      dtype='str')

## 2. Perfil epidemiológico
* total de casos;
* incidência média anual;
* total de MG;
* descritivo de MG no Estado
* total de MG por Região
* proporção de casos MG entre crianças e idosos nos municípios e nas Regiões;
* distribuição municipal.

In [14]:
# Total de casos e incidência média anual
print(
    'Total de casos:', df['TOTAL_CASOS'].sum(),
    '\n',
    'Incidência média anual:', (df['TOTAL_CASOS'].sum()/6).round(1)
)

Total de casos: 239086 
 Incidência média anual: 39847.7


In [15]:
# Total de MG e Descritivo MG no ESP
print(
    'Total de MG:', df['MG'].sum(), '\n\n',
    'Descritivo de MG no ESP:', df['MG'].describe()
)

Total de MG: 8990 

 Descritivo de MG no ESP: count    645.000000
mean      13.937984
std       31.633222
min        0.000000
25%        2.000000
50%        5.000000
75%       13.000000
max      400.000000
Name: MG, dtype: float64


In [16]:
# Top 10 mais MG
df.loc[:, ['MUNI', 'MG']].nlargest(10, 'MG')

,MUNI,MG
546,RIBEIRAO PRETO,400
635,SAO JOSE DO RIO PRETO,329
441,ARACATUBA,266
614,CAMPINAS,183
590,SOROCABA,178
611,SAO PAULO,176
472,JAU,159
466,BEBEDOURO,148
443,BIRIGUI,147
642,PIRACICABA,143


In [17]:
# Total MG e Proporção por Região

temporaria = (
    df.groupby('REGIAO', as_index=False)
      .agg(
          TOTAL_MG=('MG', 'sum'),
          TOTAL_CASOS=('TOTAL_CASOS', 'sum')
      )
)
temporaria['PROP_MG'] = (
    temporaria['TOTAL_MG'] / temporaria['TOTAL_CASOS'] * 100
).round(1)

dfReg = dfReg.merge(
    temporaria,
    on='REGIAO',
    how='left'
)

dfReg

,REGIAO,N_PESA,N_MUNICIPIOS,POP_REG,COB_PESA,POP_POR_PESA,TOTAL_MG,TOTAL_CASOS,PROP_MG
0,ARACATUBA,15,40,778764,37.5,51917.6,999,33119,3.0
1,ARARAQUARA,7,24,1017893,29.2,145413.3,284,10748,2.6
2,BAIXADA SANTISTA,8,9,1850396,88.9,231299.5,11,69,15.9
3,BARRETOS,6,18,438357,33.3,73059.5,467,8448,5.5
4,BAURU,25,68,1773654,36.8,70946.2,927,17939,5.2
5,CAMPINAS,13,42,4733979,31.0,364152.2,641,18492,3.5
6,FRANCA,9,22,700936,40.9,77881.8,371,8230,4.5
7,GRANDE SAO PAULO,16,39,21498539,41.0,1343658.7,390,6922,5.6
8,MARILIA,8,62,1133065,12.9,141633.1,490,15507,3.2
9,PIRACICABA,9,26,1590372,34.6,176708.0,366,17872,2.0


In [18]:
# Proporção de MG por faixa etaria por município

# Proporção casos MG em crianças até 10 anos
df['PROP_CASOS_MG_10'] = (df['MG_10']/df['TOTAL_10A']*100).round(1)

# Proporção casos MG em pessoas de 11 a 59 anos
df['PROP_CASOS_MG_11A59'] = (df['MG_11A59']/df['TOTAL_11A59']*100).round(1)

# Proporção casos MG em pessoas com 60 anos ou mais
df['PROP_CASOS_MG_60'] = (df['MG_60']/df['TOTAL_60']*100).round(1)

In [19]:
# Describe por municipio
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,645.0,322.000000,186.339743,0.000000,161.000000,322.000000,483.000000,6.440000e+02
ACESSO_LOCAL,645.0,0.341085,0.474442,0.000000,0.000000,0.000000,1.000000,1.000000e+00
MULTIPLO_PESA,645.0,0.018605,0.135229,0.000000,0.000000,0.000000,0.000000,1.000000e+00
LAT_MUNI,645.0,-21.187495,4.659167,-25.010461,-23.006662,-22.115854,-21.096962,0.000000e+00
LON_MUNI,645.0,-46.533098,10.066311,-53.054797,-49.998726,-48.480188,-47.153221,0.000000e+00
LAT_PESA,645.0,-21.182265,4.657134,-25.011672,-22.983918,-22.195130,-21.061391,0.000000e+00
LON_PESA,645.0,-46.526231,10.062041,-52.949023,-49.978739,-48.496295,-47.175715,0.000000e+00
DISTANCIA,425.0,27.174922,13.572230,4.300000,17.800000,24.700000,34.200000,8.440000e+01
TEMPO,425.0,27.376395,11.996764,6.800000,19.000000,25.200000,34.300000,7.080000e+01
IBGE,645.0,352869.334884,1670.029961,350010.000000,351460.000000,352870.000000,354320.000000,3.557300e+05


In [20]:
# Proporção de MG por faixa etaria por Região

temporaria = (df.groupby('REGIAO')
 .agg(
     MG_10 = ('MG_10', 'sum'),
     MG_11A59 = ('MG_11A59', 'sum'),
     MG_60 = ('MG_60', 'sum'),
     TOTAL_10A = ('TOTAL_10A', 'sum'),
     TOTAL_11A59 = ('TOTAL_11A59', 'sum'),
     TOTAL_60 = ('TOTAL_60', 'sum')
).assign(
    PROP_CASOS_MG_10 = lambda x: ((x['MG_10'] / x['TOTAL_10A']) * 100).round(1), 
    PROP_CASOS_MG_11A59 = lambda x: ((x['MG_11A59'] / x['TOTAL_11A59']) * 100).round(1),
    PROP_CASOS_MG_60 = lambda x: ((x['MG_60'] / x['TOTAL_60']) * 100).round(1)))

dfReg = dfReg.merge(
    temporaria,
    on='REGIAO',
    how='left'
)

In [21]:
# Describe Regional
dfReg.describe().T

,count,mean,std,min,25%,50%,75%,max
N_PESA,17.0,1.294118e+01,5.493981e+00,6.0,8.0,13.0,16.0,25.0
N_MUNICIPIOS,17.0,3.794118e+01,2.315294e+01,9.0,22.0,39.0,45.0,102.0
POP_REG,17.0,2.690410e+06,4.962544e+06,284892.0,778764.0,1501111.0,1850396.0,21498539.0
COB_PESA,17.0,4.142941e+01,2.211226e+01,12.9,31.0,35.6,41.0,100.0
POP_POR_PESA,17.0,1.994847e+05,3.061770e+05,18992.8,73059.5,122344.9,176708.0,1343658.7
TOTAL_MG,17.0,5.288235e+02,3.679817e+02,9.0,366.0,457.0,657.0,1471.0
TOTAL_CASOS,17.0,1.406388e+04,1.074823e+04,69.0,7124.0,10748.0,18492.0,41335.0
PROP_MG,17.0,5.041176e+00,3.250011e+00,2.0,3.2,4.4,5.6,15.9
MG_10,17.0,2.620588e+02,2.043629e+02,1.0,138.0,214.0,337.0,746.0
MG_11A59,17.0,2.134706e+02,1.383524e+02,6.0,117.0,196.0,290.0,557.0


## 3. Integração entre demanda e acesso
* casos e MG por faixas de tempo;
* casos e MG por faixas de distância;
* mapas de demanda versus acessibilidade;
* municípios com alta incidência e maior tempo de acesso.

In [22]:
# Casos por faixas de tempo nos municípios

(df.loc[df['CAT_TEMPO']!='Local',:].groupby('CAT_TEMPO')['TOTAL_CASOS'].sum().reset_index()
 .assign(PROP_CASOS = lambda x: (x['TOTAL_CASOS']/x['TOTAL_CASOS'].sum()*100).round(1)))


,CAT_TEMPO,TOTAL_CASOS,PROP_CASOS
0,31–60 min,22187,31.6
1,>60 min,496,0.7
2,≤30 min,47605,67.7


In [23]:
# Casos por faixas de tempo nas Regiões
df.loc[df['CAT_TEMPO']!='Local',:].groupby(['REGIAO','CAT_TEMPO'])['TOTAL_CASOS'].sum().reset_index()


,REGIAO,CAT_TEMPO,TOTAL_CASOS
0,ARACATUBA,31–60 min,726
1,ARACATUBA,≤30 min,4076
2,ARARAQUARA,31–60 min,1968
3,ARARAQUARA,≤30 min,1776
4,BAIXADA SANTISTA,≤30 min,10
5,BARRETOS,31–60 min,568
6,BARRETOS,≤30 min,1865
7,BAURU,31–60 min,2996
8,BAURU,>60 min,117
9,BAURU,≤30 min,2344


## 4. Comparação temporal (confirmar)
* variação 2019 a 2024
* variação na incidência;
* variação na proporção MG;
* mudança na concentração espacial;
* estabilidade ou alteração das faixas de acesso.

In [24]:
print(
    df.columns, '\n',
    dfReg.columns
)

Index(['Unnamed: 0', 'ACESSO_LOCAL', 'MULTIPLO_PESA', 'GVE', 'PESA', 'MUNI',
       'OBSERVACOES', 'LAT_MUNI', 'LON_MUNI', 'LAT_PESA', 'LON_PESA',
       'DISTANCIA', 'TEMPO', 'IBGE', 'POP10', 'POP11A59', 'POP60', 'POP_GERAL',
       'TOTAL_CASOS', 'TOTAL_10A', 'TOTAL_11A59', 'TOTAL_60', 'LEVE', 'MG',
       'MG_ORIGINAL', 'LEVE_10', 'MG_10', 'LEVE_11A59', 'MG_11A59', 'LEVE_60',
       'MG_60', 'TOTAL_AMPOLAS', 'OBITOS', 'OBITOS_10', 'OBITOS_11A59',
       'OBITOS_60', 'N_PESA', 'REGIAO', 'MUNI.1', '_merge', 'CAT_TEMPO',
       'PROP_POP_0A10', 'PROP_POP_60MAIS', 'LOG_POP', 'PROP_MG',
       'INCID_MEDIA_GERAL', 'INC_MEDIA_ANUAL', 'INCID_0A10', 'INCID_60MAIS',
       'PROP_CASOS_0A10', 'PROP_CASOS_60MAIS', 'TAXA_MG_100MIL', 'CLASS_IG',
       'SORO_IG', 'EVOL_IG', 'PROP_CLASS_IGN', 'PROP_SORO_IGN',
       'PROP_EVOL_IGN', 'BAIXO_N', 'TEMPO_50', 'CARGA_MG', 'PROP_CASOS_MG_10',
       'PROP_CASOS_MG_11A59', 'PROP_CASOS_MG_60'],
      dtype='str') 
 Index(['REGIAO', 'N_PESA', 'N_MUNICIPIO